# A tiny quantitative-finance notebook

Generate a toy price history, inspect log returns, estimate annualized parameters, and simulate a GBM. All data are synthetic and seeded for repeatability.

## 1. Generate a price series

The reusable calculations live in `src/market_demo.py`; the notebook focuses on the sequence and displays.

In [ ]:
import sys
sys.path.insert(0, "src")

import matplotlib.pyplot as plt
from market_demo import generate_prices, log_returns, estimate_parameters, simulate_gbm

prices = generate_prices()
returns = log_returns(prices)
prices.head()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(prices.index, prices, color="#2878B5")
ax.set(title="Synthetic daily closing price", xlabel="Date", ylabel="Price")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 2. Calculate and inspect log returns

For consecutive prices, the daily continuously compounded return is \(r_t = \log(P_t)-\log(P_{t-1})\).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(returns.index, returns, color="#59A14F", linewidth=0.65)
axes[0].set(title="Daily log returns", xlabel="Date", ylabel="Return")
axes[0].axhline(0, color="black", linewidth=0.7)
axes[1].hist(returns, bins=45, density=True, color="#59A14F", alpha=0.75)
axes[1].set(title="Return distribution", xlabel="Daily log return", ylabel="Density")
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
estimates = estimate_parameters(returns)
print(f"Estimated annualized drift:      {estimates['drift']:.1%}")
print(f"Estimated annualized volatility: {estimates['volatility']:.1%}")

## 3. Simulate and compare

The band is the 5th–95th percentile across simulated paths, not a real-world confidence interval.

In [ ]:
paths = simulate_gbm(
    initial_price=float(prices.iloc[0]),
    n_days=len(returns),
    drift=estimates["drift"],
    volatility=estimates["volatility"],
)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(range(len(prices)), prices.to_numpy(), color="#2878B5", label="Synthetic history")
ax.fill_between(paths.index, paths.quantile(0.05, axis=1), paths.quantile(0.95, axis=1),
                color="#F28E2B", alpha=0.2, label="Simulated 5th–95th percentile")
ax.plot(paths.index, paths.median(axis=1), color="#E15759", label="Simulated median")
ax.set(title="Synthetic history and calibrated GBM paths", xlabel="Trading day", ylabel="Price")
ax.legend(frameon=False)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()